In [2]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical   
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense 
from tensorflow.keras.callbacks import TensorBoard
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score


In [4]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

In [5]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  #Color conversion BGR to RGB
    image.flags.writeable = False                   #Image is no longer writeable

    results = model.process(image)                  #Make prediction

    image.flags.writeable = True                    #Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)  #Color conversion RGB to BGR

    return image, results

def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION)        #Draw face connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)        #Draw pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)   #Draw left hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)  #Draw right hand connections

def draw_styled_landmarks(image, results):
    #Draw face connections
    # mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION, 
    #                          mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1), 
    #                          mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
    #                          ) 
    #Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    #Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    #Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             )
    
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    # face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3) 
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)   
    return np.concatenate([pose, lh, rh])

In [6]:
# Path for exported data, numpy arrays
DATA_PATH = os.path.join("MP_Data")

# Actions that we try to detect
actions = np.array(['Hello', 'Thanks', 'I love you','Help','Sorry','Eat','Nice to meet you'])
# actions = np.array(['I love you'])



# The number of videos we want to collect per sign
no_sequences = 85
# Videos are going to be 30 frames in length
sequence_length = 45

In [7]:
for action in actions:
    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except:
            pass

In [71]:
cap = cv2.VideoCapture(0)
#Set mediapipe model
with mp_holistic.Holistic(min_detection_confidence=0.7, min_tracking_confidence=0.7) as holistic:
    for action in actions:
        # Loop through sequences aka videos
        for sequence in range(no_sequences):
            #Loop through video length aka sequence length
            for frame_num in range(sequence_length):

                #Read feed
                ret, frame = cap.read()

                #Make detections
                image, results = mediapipe_detection(frame, holistic)
                # print(results)

                #Draw landmarks
                draw_styled_landmarks(image, results)

                #Apply collection logic
                if frame_num == 0:
                    cv2.putText(image, "STARTING COLLECTION", (120,200),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, f'Collecting frames for {action} Video Number {sequence}', 
                                (15,12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255, 0), 1, cv2.LINE_AA)
                    cv2.waitKey(3000)
                else:                 
                    cv2.putText(image, f'Collecting frames for {action} Video Number {sequence}', 
                                (15,12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255, 0), 1, cv2.LINE_AA)
                # New export keypoints
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

                #Show feed
                cv2.imshow("OpenCV Feed", image)

                #Break
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    break

cap.release()
cv2.destroyAllWindows()

In [8]:
label_map = {label:num for num, label in enumerate(actions)}

In [9]:
# sequences, labels = [], []
# for action in actions:
#     for sequence in range(no_sequences):
#         window = []
#         for frame_num in range(sequence_length):
#             res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
#             window.append(res)
#         sequences.append(window)
#         labels.append(label_map[action])

In [ ]:
TARGET_LENGTH = 65
sequences, labels = [], []

for action in actions:
    for sequence in range(no_sequences):
        window = []

        sequence_path = os.path.join(DATA_PATH, action, str(sequence))

        files = sorted(
            os.listdir(sequence_path),
            key=lambda x: int(x.split('.')[0])
        )

        for file in files:
            window.append(
                np.load(os.path.join(sequence_path, file))
            )

        while len(window) < TARGET_LENGTH:
            window.append(window[-1])

        window = window[:TARGET_LENGTH]

        sequences.append(window)
        labels.append(label_map[action])

In [12]:
print(x.shape)
print(y.shape)

(595, 65, 258)


NameError: name 'y' is not defined

In [11]:
x = np.array(sequences, dtype=np.float32)
x.shape

(595, 65, 258)

In [13]:
y = to_categorical(labels).astype(int)
y

array([[1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 1]])

In [12]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=np.argmax(y, axis=1),
    random_state=42
)

In [13]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

In [14]:
model = Sequential()

model.add(LSTM(128,
               return_sequences=True,
               input_shape=(65,258)))

model.add(LSTM(256,
               return_sequences=True))

model.add(LSTM(128,
               return_sequences=False))

model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))

model.add(Dense(actions.shape[0],
                activation='softmax'))

c:\Users\amira\OneDrive\Documents\PROJECT\SIGN_LANGUAGE_DETECTION\SLD_1\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [15]:
print(x.shape)
print(y.shape)
print(x_train.shape)
print(y_train.shape)
print(np.sum(y, axis=0))
print(x.dtype)

(595, 65, 258)
(595, 7)
(476, 65, 258)
(476, 7)
[85 85 85 85 85 85 85]
float32


In [16]:
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

In [17]:
model.fit(x_train, y_train, epochs=70, callbacks=[tb_callback])
# from tensorflow.keras.callbacks import EarlyStopping

# early_stop = EarlyStopping(
#     monitor='val_categorical_accuracy',
#     patience=10,
#     restore_best_weights=True
# )

# history = model.fit(
#     x_train,
#     y_train,
#     validation_data=(x_test, y_test),
#     epochs=100,
#     callbacks=[early_stop]
# )

Epoch 1/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 16s 208ms/step - categorical_accuracy: 0.1387 - loss: 1.9379
Epoch 2/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 209ms/step - categorical_accuracy: 0.2521 - loss: 1.7742
Epoch 3/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 211ms/step - categorical_accuracy: 0.3025 - loss: 1.6469
Epoch 4/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 216ms/step - categorical_accuracy: 0.3613 - loss: 1.5087
Epoch 5/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 209ms/step - categorical_accuracy: 0.3550 - loss: 1.4029
Epoch 6/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 210ms/step - categorical_accuracy: 0.3592 - loss: 1.3403
Epoch 7/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 205ms/step - categorical_accuracy: 0.3298 - loss: 1.4011
Epoch 8/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 196ms/step - categorical_accuracy: 0.3676 - loss: 1.3103
Epoch 9/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 195ms/step - categorical_accuracy: 0.3529 - loss: 1.3256
Epoch 10/70
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 196ms/step - categorical_accuracy: 0.3508 - loss: 1.2842
Epoch 11/70
15/15 

In [18]:
res = model.predict(x_test)

4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 541ms/step


In [19]:
actions[np.argmax(res[7])]

'Hello'

In [20]:
actions[np.argmax(res[7])]

'Hello'

In [21]:
yhat = model.predict(x_test)

ytrue = np.argmax(y_test, axis=1)
ypred = np.argmax(yhat, axis=1)

print("Accuracy:", accuracy_score(ytrue, ypred))
print(multilabel_confusion_matrix(ytrue, ypred))

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 173ms/step
Accuracy: 0.9327731092436975
[[[ 99   3]
  [  1  16]]

 [[100   2]
  [  0  17]]

 [[102   0]
  [  4  13]]

 [[102   0]
  [  0  17]]

 [[102   0]
  [  2  15]]

 [[ 99   3]
  [  0  17]]

 [[102   0]
  [  1  16]]]


In [23]:
model.save("sign_language_model.h5")

In [16]:
from tensorflow.keras.models import load_model

model = load_model("sign_language_model.h5")

In [24]:
colors = [
    (245,117,16),
    (117,245,16),
    (16,117,245),
    (255,0,0),
    (255,255,0),
    (255,0,255),
    (0,255,255),
    (128,0,255),
    (0,128,255)
]

def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()

    start_y = 60
    row_height = 50
    bar_max_width = 300

    for num, prob in enumerate(res):
        y1 = start_y + num * row_height
        y2 = y1 + 30

        color = colors[num % len(colors)]

        cv2.rectangle(
            output_frame,
            (10, y1),
            (10 + int(prob * bar_max_width), y2),
            color,
            -1
        )

        cv2.putText(
            output_frame,
            f"{actions[num]}: {prob:.2f}",
            (320, y2 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    return output_frame

In [25]:
# plt.figure(figsize=(10,8))
# plt.imshow(cv2.cvtColor(prob_viz(res, actions, image, colors), cv2.COLOR_BGR2RGB))
# plt.axis("off")
# plt.show()

In [ ]:
# #New detection variable
# sequence = []
# sentence = []
# predictions = []
# threshold = 0.5

# cap = cv2.VideoCapture(0)
# #Set mediapipe model
# with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
#     while cap.isOpened():
#             #Read feed
#             ret, frame = cap.read()

#             #Make detections
#             image, results = mediapipe_detection(frame, holistic)
#             # print(results)

#             #Draw landmarks
#             draw_styled_landmarks(image, results)

#             # 2. Prediction logic
#             keypoints = extract_keypoints(results)
#             sequence.append(keypoints)
#             sequence = sequence[-30:]

#             if len(sequence) == 30:
#                 res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
#                 # print(actions[np.argmax(res)])
#                 predictions.append(np.argmax(res))

#                 if np.unique(predictions[-10:])[0] == np.argmax(res):
#                     if res[np.argmax(res)] > threshold:
#                         if len(sentence) > 0:
#                             if actions[np.argmax(res)] != sentence[-1]:
#                                 sentence.append(actions[np.argmax(res)])
#                         else:
#                             sentence.append(actions[np.argmax(res)])
        
#             if len(sentence) > 5:
#                  sentence = sentence[-5:]

#             #Viz probabilities
#             # image = prob_viz(res, actions, image, colors) #-> for real time probability calculation
                           
#             cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
#             cv2.putText(image, ' '.join(sentence), (3,30),
#                         cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

#             #Show feed
#             cv2.imshow("OpenCV Feed", image)

#             #Break
#             if cv2.waitKey(10) & 0xFF == ord('q'):
#                 break

# cap.release()
# cv2.destroyAllWindows()

In [18]:
# New detection variables

sequence = []
predictions = []

current_prediction = ""
threshold = 0.75

cap = cv2.VideoCapture(0)

with mp_holistic.Holistic(
min_detection_confidence=0.5,
min_tracking_confidence=0.5) as holistic:

    while cap.isOpened():

        ret, frame = cap.read()

        image, results = mediapipe_detection(frame, holistic)

        draw_styled_landmarks(image, results)

        keypoints = extract_keypoints(results)

        sequence.append(keypoints)

        # Use same length as training
        sequence = sequence[-65:]

        if len(sequence) == 65:

            res = model.predict(
                np.expand_dims(sequence, axis=0),
                verbose=0
            )[0]

            predicted_class = np.argmax(res)
            confidence = res[predicted_class]

            predictions.append(predicted_class)

            # Stability check
            if len(predictions) >= 10:

                recent_predictions = predictions[-10:]

                if (
                    len(set(recent_predictions)) == 1 and
                    confidence > threshold
                ):
                    current_prediction = (
                        f"{actions[predicted_class]}"
                        f" ({confidence:.2f})"
                    )

        # Bottom prediction bar
        cv2.rectangle(
            image,
            (0, image.shape[0] - 50),
            (image.shape[1], image.shape[0]),
            (245, 117, 16),
            -1
        )

        cv2.putText(
            image,
            current_prediction,
            (20, image.shape[0] - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        cv2.imshow("OpenCV Feed", image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break


cap.release()
cv2.destroyAllWindows()
